# Verificación de salida y curvas: prueba de 20 a 19 K
Lee los archivos NPZ de dos estados producidos por los cuadernos de simulación. Comprueba temperaturas, dimensiones, valores, parámetros y coherencia de las energías; genera vistas de espines y curvas por copia. No ejecuta la simulación.

En Kaggle, adjunta como entrada la salida del cuaderno de simulación y ejecuta estas celdas. Si hay varios conjuntos de salida, especifica `RUTA_DATOS` para evitar mezclar corridas.

In [ ]:
from pathlib import Path
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# None busca en /kaggle/input y /kaggle/working. También puedes escribir
# una ruta concreta, por ejemplo Path('/kaggle/input/tu-dataset/lote_01').
RUTA_DATOS = None
TEMPERATURAS_ESPERADAS = np.arange(200, 189, -1)/10  # 20.0, 19.9, ..., 19.0 K
REPLICAS_ESPERADAS = 100
MEDICIONES_ESPERADAS = 10000
ESPINES_ESPERADOS = 5425
SALIDA = Path('/kaggle/working/verificacion_salida') if Path('/kaggle').exists() else Path('verificacion_salida')
SALIDA.mkdir(parents=True, exist_ok=True)
print('Salida de informes y figuras:', SALIDA.resolve())

In [ ]:
# Identificación de archivos. Solo se admiten carpetas estado_N/T_*.npz.
raices = [Path(RUTA_DATOS)] if RUTA_DATOS else [Path('/kaggle/input'), Path('/kaggle/working')]
candidatos=[]
for raiz in raices:
    if raiz.exists():
        candidatos += [p for p in raiz.rglob('T_*.npz') if re.fullmatch(r'estado_\d+',p.parent.name)]
por_carpeta={}
for p in sorted(set(candidatos)): por_carpeta.setdefault(p.parent,[]).append(p)
print('Carpetas de estados detectadas:')
for carpeta, archivos in por_carpeta.items(): print(f'  {carpeta}: {len(archivos)} archivos')
if not por_carpeta: raise FileNotFoundError('No se encontraron archivos. Adjunta la salida o define RUTA_DATOS.')
if len(por_carpeta) != 2:
    raise ValueError('Se esperaban exactamente dos carpetas de estado. Define RUTA_DATOS para seleccionar una corrida.')


## Comprobación de integridad
Cada archivo se abre y se comprueban los 100 vectores de espines, las seis series medidas y los metadatos. También se contrastan las temperaturas con los 11 valores esperados y se verifica que la energía total coincida con la suma de sus componentes.

In [ ]:
SERIES=('E','M','E_exc','E_an','E_dm','E_hex')
META=('T','Jex','kDM','Kan1','Hex','gamma','Rd','So','kB','Eo','term','med','semilla','keys','formato')
resultados=[]; datos={}; problemas=[]
for carpeta, archivos in sorted(por_carpeta.items()):
    estado=carpeta.name; filas=[]; firmas=[]
    for archivo in archivos:
        errores=[]; registro={'estado':estado,'archivo':str(archivo),'T':np.nan}
        try:
            with np.load(archivo,allow_pickle=False) as d:
                faltan=set(('spins','mascara')+SERIES+META)-set(d.files)
                if faltan: raise ValueError('Faltan claves: '+', '.join(sorted(faltan)))
                T=float(d['T']); registro['T']=T
                if archivo.name != f'T_{T:.1f}.npz': errores.append('nombre y temperatura difieren')
                mascara=d['mascara'].astype(bool); spins=d['spins']; valores={k:d[k] for k in SERIES}
                if mascara.shape != (5,39,39) or int(mascara.sum()) != ESPINES_ESPERADOS: errores.append('máscara o número de espines incorrecto')
                if spins.shape != (REPLICAS_ESPERADAS,ESPINES_ESPERADOS,3): errores.append(f'forma de spins: {spins.shape}')
                if not np.isfinite(spins).all(): errores.append('spins no finitos')
                max_norma=float(np.max(np.abs(np.linalg.norm(spins,axis=-1)-float(d['So'])))); registro['error_max_norma']=max_norma
                if max_norma>1e-3: errores.append('norma de espines fuera de tolerancia')
                for k,v in valores.items():
                    if v.shape != (MEDICIONES_ESPERADAS,REPLICAS_ESPERADAS): errores.append(f'{k}: forma {v.shape}')
                    if not np.isfinite(v).all(): errores.append(k+' tiene valores no finitos')
                if int(d['med']) != MEDICIONES_ESPERADAS: errores.append('metadato med distinto del esperado')
                diferencia=np.max(np.abs(valores['E']-(valores['E_exc']+valores['E_an']+valores['E_dm']+valores['E_hex'])))
                registro['max_error_componentes']=float(diferencia)
                if not np.allclose(valores['E'],valores['E_exc']+valores['E_an']+valores['E_dm']+valores['E_hex'],rtol=1e-5,atol=1e-6): errores.append('E no coincide con suma de componentes')
                firma=tuple(float(d[x]) for x in ('Jex','kDM','Kan1','Hex','gamma','Rd','So','kB','Eo'))+(int(d['term']),int(d['med']),int(d['semilla']))
                firmas.append(firma); datos[(estado,round(T,1))]=archivo
                registro.update(dict(zip(('Jex','kDM','Kan1','Hex','gamma','Rd','So','kB','Eo','term','med','semilla'),firma)))
        except Exception as exc: errores.append(f'{type(exc).__name__}: {exc}')
        registro['resultado']='OK' if not errores else '; '.join(errores)
        resultados.append(registro); filas.append(registro)
        if errores: problemas.append(f'{archivo}: {registro["resultado"]}')
    encontradas=[round(x['T'],1) for x in filas if np.isfinite(x['T'])]
    esperadas=set(np.round(TEMPERATURAS_ESPERADAS,1)); faltan=sorted(esperadas-set(encontradas),reverse=True); extras=sorted(set(encontradas)-esperadas,reverse=True)
    if faltan: problemas.append(f'{estado}: faltan temperaturas {faltan}')
    if extras: problemas.append(f'{estado}: temperaturas fuera de la prueba {extras}')
    if len(encontradas)!=len(set(encontradas)): problemas.append(f'{estado}: temperaturas duplicadas')
    if firmas and any(f!=firmas[0] for f in firmas[1:]): problemas.append(f'{estado}: parámetros o semilla cambian entre temperaturas')
    print(f'{estado}: {len(encontradas)} archivos; faltantes={faltan}; extras={extras}',flush=True)
verificacion=pd.DataFrame(resultados).sort_values(['estado','T'],ascending=[True,False]); display(verificacion)
verificacion.to_csv(SALIDA/'verificacion_archivos.csv',index=False)
(SALIDA/'problemas.json').write_text(json.dumps(problemas,indent=2,ensure_ascii=False))
if problemas:
    print('PROBLEMAS DETECTADOS:'); [print('-',p) for p in problemas]
else: print('VERIFICACIÓN CORRECTA: 11 temperaturas completas para ambos estados.')

## Texturas representativas
Se muestra la componente vertical de los espines en la capa central para tres copias y tres temperaturas disponibles. Cada fila es un estado; las columnas identifican temperatura y copia. No son promedios ni demuestran por sí solas convergencia.

In [ ]:
estados=sorted({e for e,t in datos});
if problemas: print('Atención: se grafican archivos legibles, pero hay problemas en la verificación.')
for estado in estados:
    ts=sorted([t for e,t in datos if e==estado],reverse=True)
    if not ts: continue
    seleccion=[ts[0],ts[len(ts)//2],ts[-1]]; copias=[0,49,99]
    fig,axes=plt.subplots(len(seleccion),len(copias),figsize=(10,9),squeeze=False)
    for i,t in enumerate(seleccion):
        with np.load(datos[(estado,t)],allow_pickle=False) as d:
            mask=d['mascara'].astype(bool); spins=d['spins']; capa=mask.shape[0]//2
            for j,copia in enumerate(copias):
                red=np.full((*mask.shape,3),np.nan,dtype=np.float32); red[mask]=spins[copia]
                im=axes[i,j].imshow(red[capa,:,:,2],origin='lower',cmap='coolwarm',vmin=-1,vmax=1)
                axes[i,j].set_title(f'{t:.1f} K · copia {copia+1}'); axes[i,j].axis('off')
    fig.suptitle(f'{estado}: componente $S_z$, capa central'); fig.colorbar(im,ax=axes.ravel().tolist(),label='$S_z$',shrink=.75)
    fig.savefig(SALIDA/f'{estado}_texturas.png',dpi=160,bbox_inches='tight'); plt.show()

## Curvas de las 100 copias
Para cada temperatura y copia se promedia la energía y magnetización en los 10.000 barridos. Las fluctuaciones se calculan dentro de cada copia: $c=\mathrm{Var}(H)/(N k_B T^2)$ y $\chi=N\,\mathrm{Var}(m)/(k_B T)$, con energía física $H=E\times E_0$ y $m=|\sum_i S_i|/N$. Se usa varianza con divisor $n$, como en el código de simulación. Esta susceptibilidad corresponde a esa definición de magnetización; no debe confundirse con una componente con signo.

Las gráficas muestran las 100 curvas y su media. No son intervalos de confianza; el análisis de autocorrelación y remuestreo queda pendiente.

In [ ]:
filas=[]
for (estado,T),archivo in sorted(datos.items()):
    try:
        with np.load(archivo,allow_pickle=False) as d:
            eo=float(d['Eo']); kb=float(d['kB']); N=int(d['mascara'].sum())
            E=d['E'].astype(np.float64)*eo; M=d['M'].astype(np.float64)
            comp={k:d[k].astype(np.float64).mean(axis=0)*eo/N for k in ('E_exc','E_an','E_dm','E_hex')}
            e=E.mean(axis=0)/N; m=M.mean(axis=0)
            c=E.var(axis=0)/(N*kb*T*T); chi=N*M.var(axis=0)/(kb*T)
            for r in range(E.shape[1]):
                filas.append(dict(estado=estado,T=T,copia=r+1,energia_por_espin=e[r],magnetizacion=m[r],calor_especifico=c[r],susceptibilidad=chi[r],**{k:v[r] for k,v in comp.items()}))
    except Exception as exc: print('No se pudo calcular:',archivo,exc)
curvas=pd.DataFrame(filas)
if curvas.empty: raise RuntimeError('No hay datos legibles para graficar.')
curvas.sort_values(['estado','copia','T'],inplace=True); curvas.to_csv(SALIDA/'curvas_100_copias.csv',index=False)
display(curvas.head())

In [ ]:
variables=[('energia_por_espin','Energía por espín'),('magnetizacion','Magnetización'),('calor_especifico','Calor específico'),('susceptibilidad','Susceptibilidad'),('E_exc','Intercambio por espín'),('E_an','Anisotropía por espín'),('E_dm','DMI por espín'),('E_hex','Campo por espín')]
for estado in sorted(curvas.estado.unique()):
    sub=curvas[curvas.estado==estado]; fig,axs=plt.subplots(4,2,figsize=(14,15),sharex=True)
    for ax,(var,titulo) in zip(axs.flat,variables):
        for _,rep in sub.groupby('copia'):
            rep=rep.sort_values('T'); ax.plot(rep.T if False else rep['T'],rep[var],color='steelblue',alpha=.12,lw=.6)
        media=sub.groupby('T')[var].mean().sort_index(); ax.plot(media.index,media.values,color='black',lw=2,label='Media entre copias')
        ax.set_title(titulo); ax.grid(alpha=.2); ax.legend()
    for ax in axs[-1]: ax.set_xlabel('Temperatura (K)')
    fig.suptitle(f'{estado}: 100 copias y media',fontsize=15); fig.tight_layout()
    fig.savefig(SALIDA/f'{estado}_curvas.png',dpi=160); plt.show()
print('Archivos generados en',SALIDA)